# 03 - Özellik Mühendisliği

## Amaç ve Kapsam

Bu notebook, tezin yapay zeka modelleri (Random Forest, XGBoost, LightGBM, CNN, LSTM) için en kritik **veri dönüşüm ve özellik türetme (Feature Engineering)** sürecini temsil etmektedir. Ham verilerin salt halleriyle tahmingücünün (predictive validity) kısıtlı olması nedeniyle, verinin matematiksel veya zamansal ilişkileri deşifre edilerek doğrudan modellerin öğrenebileceği sinyallere dönüştürülür. İş sürelerini (runtime) yüksek bir doğrulukla tahmin edecek modellerin başarısı, tümüyle bu matris mimarisinin kalitesine dayanır.

Bu aşamada gerçekleştirilen temel dönüşüm operasyonları ve tez mimarisine katkıları şunlardır:

1. **Sweep-Line Arka Plan Yük Hesaplaması:** Kuyruk teorisinin temel ilkesi olan "sistem doluluğu iş süresini etkiler" prensibini modellemek üzere, işlerin varış anındaki (arrival) anlık küme CPU/GPU streslerinin gelişmiş bir Sweep-Line algoritması vasıtasıyla dinamik olarak türetilmesi.
2. **Gündüz/Gece Döngüsel Sinyalleri (Temporal Features):** `arrival_time` loglarından günün saati fazı ve iz-göreli gün indeksi (0, 1, 2, ...) çıkarılarak, ritimsel submission paternlerinin (özellikle derin öğrenme ağları için) bir patern olarak kullanılabilmesi. Bu sinyallerin takvim karşılığı yoktur: izin genel yayınında toplama tarihi açıklanmadığı için `day_of_week`, ilk gelişten itibaren sayılan bir gün sayacıdır; haftanın günü *değildir*.
3. **Yüksek Kardinaliteli Kategorik İşlem (Encoding):** `user` (kullanıcı) ve `gpu_type` donanım özniteliklerinin One-Hot (ve LightGBM için Native) teknikleriyle istatistiksel model uzayına haritalanması; bu sayede bireysel geliştirici kullanım alışkanlıklarının süre tahminine yansıtılması.
4. **Kronolojik Test-Eğitim Ayrımı (Temporal Train/Test Split):** Makine öğrenmesinde sık görülen klasik randomize (rastgele) veri bölüşümünün aksine; modellerin gelecek zamanı tahmin etmedeki "gerçekçi" performansını (generalization capabilities) ölçebilmek için veri setinin sıkı bir zaman ekseni (kronolojik) üzerinden test ve eğitim (%80/20) alt kümelerine ayrılması.

Elde edilen nihai (ve zenginleştirilmiş) eğitim/test kütükleri disk modüllerine kaydedilerek doğrudan Notebook 04 (Model Eğitimi) ortamına eksiksiz bir şekilde entegre edilmektedir.


In [1]:
# ── 0. Environment & Path Setup ──────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Anchor on a marker that only exists at the repository root, so the notebook
# works regardless of the directory the kernel was started from. The previous
# parents[1] form silently resolved one level short when the working directory
# was notebooks/ rather than notebooks/<lang>/.
def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    return start.parents[1]

PROJECT_ROOT = _find_project_root(Path().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"[Setup] Project root : {PROJECT_ROOT}")
print(f"[Setup] Python path  : {sys.executable}")

[Setup] Project root : /Users/hasanugurcelebi/Thesis/alibaba-gpu-runtime-prediction-and-scheduling
[Setup] Python path  : /Users/hasanugurcelebi/Thesis/alibaba-gpu-runtime-prediction-and-scheduling/venv/bin/python


> **Çalışma Ortamı ve Kök Dizin:** Proje kök dizini dinamik olarak çözümlenerek `sys.path` üzerine eklenmiştir. Bu sayede modüller arası mutlak yollar korunur.

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loading import load_sample
from src.feature_engineering import (
    build_job_table_from_sample,
    add_temporal_features,
    add_categorical_features,
    add_cluster_utilization_features,
    build_feature_matrix,
)

# Unified visualisation theme
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.family": "DejaVu Sans",
    # ── Publication-grade output (figure/table audit) ──────────────────────
    # scripts/export_thesis_results.py scrapes the notebook's own inline PNG,
    # so the INLINE dpi is what reaches the thesis. At the 100 dpi default
    # these exported at 147-280 ppi, under the 300 ppi publisher floor.
    "figure.dpi": 200,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    # Figures are now authored at roughly the width they are PRINTED at
    # (\textwidth = 6.10 in in thesis.cls: a4paper, 3.5cm/2cm margins), so the
    # downscale is ~0.8x rather than the ~0.34x that made in-figure text print
    # at 2.5-4.8 pt -- under the ~6 pt legibility floor. At these sizes the
    # smallest label prints at ~6.4 pt and body text at ~7.2 pt.
    # Figures are authored on a generous canvas (10-18 in) and printed into a
    # 6.10-in column, roughly a 0.35-0.55x reduction. Shrinking the canvas to
    # match the print width made every panel cramped, so the canvas stays and
    # the TYPE is scaled instead: at 15 pt a 14-in figure prints at 6.5 pt,
    # above the ~6 pt legibility floor, while still looking uncrowded at
    # authoring size.
    "font.size": 15,
    "axes.titlesize": 17,
    "axes.labelsize": 15,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "figure.titlesize": 19,
})

print("[Setup] All imports OK.")


> **Kütüphane İçe Aktarımları:** Veri işleme, görselleştirme ve sistem işlemleri için gereken tüm temel Python ve bilimsel kütüphaneler başarıyla ortama dahil edilmiştir.

In [3]:
# ── 2. Load raw data ──────────────────────────────────────────────────────────
print("[Step 1] Loading raw dataset...")
raw_df = load_sample(which="main")

print(f"[Step 1] Shape : {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
raw_df.head()

[Step 1] Loading raw dataset...
[Step 1] Shape : 100,000 rows × 8 columns


,job_id,num_inst,submit_time,num_cpu,num_gpu,gpu_type,duration,user
0,0,1.0,0,2.0,0.00,CPU,34,d4d51aca8806
1,1,12.0,3,6.0,0.25,T4,15748,d4d51aca8806
2,2,1.0,5,6.0,1.00,MISC,84,a8192d6b0ae9
3,3,1.0,21,18.0,1.00,T4,46,c7152ce0fec1
4,4,1.0,21,6.0,1.00,MISC,80,7b76597f4283


> **Ham Veri Yükleme:** 100.000 satırlık ham GPU görev dağılım kütüğü diske okunmuştur. Bu veri, modellerin eğitileceği temel havuzu oluşturmaktadır.

In [4]:
# ── 3. Schema inspection ──────────────────────────────────────────────────────
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   job_id       100000 non-null  int64  
 1   num_inst     100000 non-null  float64
 2   submit_time  100000 non-null  int64  
 3   num_cpu      100000 non-null  float64
 4   num_gpu      100000 non-null  float64
 5   gpu_type     100000 non-null  object 
 6   duration     100000 non-null  int64  
 7   user         100000 non-null  object 
dtypes: float64(3), int64(3), object(2)
memory usage: 6.1+ MB


> **Şematik İnceleme:** Ham verinin bellek tüketimi (6.1+ MB) ve 8 adet temel özniteliğinin (`job_id`, `gpu_type` vb.) veri tipleri doğrulanmıştır.

In [5]:
# ── 4. Normalise raw data ─────────────────────────────────────────────────────
print("[Step 2] Building canonical job table...")
# include_cpu_only=True mirrors prepare_features_for_model(), which is what
# notebooks 04 and 05 actually train and simulate on. The sweep-line below has
# to see EVERY job holding resources -- CPU-only jobs included -- or the
# background-load features this chapter illustrates are not the features the
# models are given (leakage-4 / robustness-11). GPU jobs are selected after the
# sweep-line, not before.
job_df = build_job_table_from_sample(raw_df, time_unit="s", include_cpu_only=True)

print(f"[Step 2] Jobs incl. CPU-only : {len(job_df):,}  (removed {len(raw_df)-len(job_df):,} invalid rows)")
job_df.head()

[Step 2] Building canonical job table...
[Step 2] Jobs incl. CPU-only : 100,000  (removed 0 invalid rows)


,job_id,arrival_time,arrival_sec,job_runtime,gpu_demand,user,gpu_type,num_inst,num_cpu
0,0,1970-01-01 00:00:00,0.0,34.0,0.00,d4d51aca8806,CPU,1.0,2.0
1,1,1970-01-01 00:00:03,3.0,15748.0,0.25,d4d51aca8806,T4,12.0,6.0
2,2,1970-01-01 00:00:05,5.0,84.0,1.00,a8192d6b0ae9,MISC,1.0,6.0
3,3,1970-01-01 00:00:21,21.0,46.0,1.00,c7152ce0fec1,T4,1.0,18.0
4,4,1970-01-01 00:00:21,21.0,80.0,1.00,7b76597f4283,MISC,1.0,6.0


> **Veri Normalizasyonu:** Ham verideki Unix zaman damgaları ve kategorik formatlar, analizlerin tam yapılabilmesi için standart panda tiplerine dönüştürülmüştür.

In [6]:
# ── 5. Shape check ────────────────────────────────────────────────────────────
print(f"[Step 2] Feature matrix shape : {job_df.shape}")

[Step 2] Feature matrix shape : (100000, 9)


> **Veri Filtreleme ve Şekil Kontrolü:** Tablo bu noktada hâlâ 100.000 kaydın tamamını içermektedir — `include_cpu_only=True`, `num_gpu == 0` olan 17.816 yalnızca-CPU işini bilinçli olarak korumaktadır; böylece aşağıdaki sweep-line, bir işin gerçekten içine geldiği yükü ölçmektedir. Ham izde eksik veya pozitif olmayan süre bulunmadığından burada hiçbir kayıt elenmemektedir; tezin modellediği 82.184 GPU işine daralma, sweep-line'dan sonra, aşağıdaki hücrede gerçekleşmektedir.


In [7]:
# ── 6. Cluster utilisation features ──────────────────────────────────────────
print("[Step 3] Computing cluster utilisation features (sweep-line)...")
job_df = add_cluster_utilization_features(job_df)

# The sweep-line has now seen the CPU-only jobs; restrict to GPU jobs, which
# are what the thesis models. Same order as prepare_features_for_model().
_n_all = len(job_df)
job_df = job_df[job_df["gpu_demand"] > 0].copy().reset_index(drop=True)
print(f"[Step 3] Restricted to GPU jobs : {len(job_df):,} of {_n_all:,}")

print(f"[Step 3] New columns : cluster_load_cpu, cluster_load_gpu, active_job_count")
job_df[["arrival_sec", "gpu_demand", "cluster_load_gpu",
        "active_job_count"]].head()

[Step 3] Computing cluster utilisation features (sweep-line)...
[Step 3] Restricted to GPU jobs : 82,184 of 100,000
[Step 3] New columns : cluster_load_cpu, cluster_load_gpu, active_job_count


,arrival_sec,gpu_demand,cluster_load_gpu,active_job_count
0,3.0,0.25,0.00,1
1,5.0,1.00,0.25,2
2,21.0,1.00,2.25,4
3,21.0,1.00,2.25,4
4,36.0,1.00,3.25,4


> **Sweep-Line Olay Taraması (Küme Doluluk Oranları):**
> Tahmin modellerinin çevresel donanım krizlerine karşı duyarlılığını artırmak amacıyla, her bir işin sisteme girdiği andaki küme doluluğu (cluster contention) hesaplanmıştır. 
>
> Kesişen binlerce işin üst üste bindiği (overlap) bu anlık yükleri statik O(N²) döngülerle bulmak imkânsız olduğundan, tüm submit ve end logları kronolojik bir evreye dizilerek **Olay Tabanlı Tarama Çizgisi (Sweep-line Algorithm)** ile taranmıştır. Bu sayede her işin geliş anındaki **mevcut aktif CPU/GPU yükü ve devam eden diğer işlerin sayısı**, veri sızıntısı (data leakage) yaratılmadan modele doğrudan enjekte edilmiştir.


In [ ]:
# ── 7. Active job count over time ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

axes[0].plot(job_df["arrival_sec"] / 86400, job_df["active_job_count"],
             alpha=0.6, lw=0.8, color="royalblue", label="Active Jobs")
axes[0].set_ylabel("Background Job Count")
axes[0].set_title("Cluster State: Background Job Count Over Time")
# One series per panel, so a one-entry legend only repeated the title -- and in
# the lower panel it sat on top of a late-trace peak. Dropped both.
axes[0].grid(True, alpha=0.3)

axes[1].plot(job_df["arrival_sec"] / 86400, job_df["cluster_load_gpu"],
             alpha=0.6, lw=0.8, color="darkorange", label="GPU Demand")
axes[1].set_ylabel("Background GPU Demand (GPUs)")
axes[1].set_xlabel("Elapsed Time (days)")
axes[1].set_title("Cluster State: Background GPU Demand Over Time")
axes[1].grid(True, alpha=0.3)
# Headroom so the peaks are not flush against the frame.
for _ax in axes:
    _lo, _hi = _ax.get_ylim()
    _ax.set_ylim(_lo, _hi * 1.08)

plt.tight_layout()
plt.show()

> **Şekil 1 — 8 Günlük Aktif İş ve Kümülatif GPU Talebi:**
>
> Yukarıdaki zaman dalgaları (time-series), hesaplanan Sweep-Line verisinin **~8 günlük bir anonim iz (trace)** üzerindeki dinamiklerini somutlaştırmaktadır. İlk grafik sistemdeki devam eden **Aktif İş Sayısının** nasıl 400 bandından fırlayarak şiddetli **sıçramalarla (bursts)** 1.000 sınırını aştığını göstermektedir. İkinci grafikteki **Kümülatif GPU Talebi** de bu dalgalanmaya eşlik ederek anlık 800 GPU sömürüsüne ulaşmaktadır.
>
> Bu aşırı değişken sistem stresi; zamanlama algoritmalarının salt-statik metriklerle neden yetersiz kaldığını görsel olarak kanıtlamakta, yapay zekanın bu anlık (point-in-time) arka plan dar boğazlarını tahmin denklemlerine dahili özellik (feature) olarak almasını zorunlu kılmaktadır.


In [9]:
# ── 8. Temporal features ──────────────────────────────────────────────────────
# IMPORTANT: add_temporal_features needs arrival_time — call BEFORE any drop.
print("[Step 5] Extracting temporal features from arrival_time...")
job_df = add_temporal_features(job_df)

print("[Step 5] New columns : hour_of_day, day_of_week")
job_df[["arrival_time", "hour_of_day", "day_of_week"]].head()

[Step 5] Extracting temporal features from arrival_time...
[Step 5] New columns : hour_of_day, day_of_week


,arrival_time,hour_of_day,day_of_week
0,1970-01-01 00:00:03,0,0
1,1970-01-01 00:00:05,0,0
2,1970-01-01 00:00:21,0,0
3,1970-01-01 00:00:21,0,0
4,1970-01-01 00:00:36,0,0


> **Zamansal Sinyaller (Temporal Features):**
> İş gelişlerinin salt rastgelelikten ziyade bir "çalışma ritmine" sahip olduğu düşünülerek, `arrival_time` üzerinden `hour_of_day` (günün saati fazı) ve `day_of_week` (iz-göreli gün indeksi) özellikleri türetilmiştir.
>
> Her iki sinyal de iz-göreli olup takvime bağlı **değildir**. `hour_of_day`, izin başlangıcından itibaren artan bir ofsetten türetildiği ve yayın gerçek toplama tarihini açıklamadığı için mutlak hizası bilinmeyen 24 saatlik bir fazdır; buradaki "saat 9", sabah 9 olarak okunamaz. `day_of_week` ise adına rağmen ilk gelişten itibaren sayılan mutlak bir gün sayacıdır (0, 1, 2, ...) ve ~7,7 günlük bu izde bir takvim gününün alamayacağı sekiz farklı değer (0-7) alır; hafta içi/hafta sonu anlamı taşımaz ve "Pazartesi" gibi okunmamalıdır (bkz. Notebook 01'deki gün-indeksi düzeltmesi). Gerçek bir `dt.dayofweek` tam da bu nedenle reddedilmiştir: kronolojik eğitim/test bölmesinde izin 0. gününü ve 7. gününü aynı etikete katlardı.
>
> Modellenebilir olarak geriye kalan şey *göreli* ritimdir: bir gün içinde hangi saatlerin diğerlerine göre yoğun olduğu ve yükün bir iz gününden diğerine nasıl kaydığı — sıralı mimarilerin (LSTM) kullanabileceği sinyal budur. Üretilen döngüsel özellikler, bellekte yer kaplamaması adına özel olarak en verimli `int32` formatında belleğe kodlanmıştır.

In [10]:
# ── 9. Categorical encoding ───────────────────────────────────────────────────
print("[Step 6] Encoding categorical features...")
job_df = add_categorical_features(job_df)

# arrival_time is no longer needed after temporal extraction
job_df = job_df.drop(columns=["arrival_time"], errors="ignore")

print("[Step 6] Column dtypes:")
print(job_df.dtypes.to_string())

[Step 6] Encoding categorical features...
[Step 6] Column dtypes:
job_id                 int64
arrival_sec          float64
job_runtime          float64
gpu_demand           float64
user                category
gpu_type            category
num_inst             float64
num_cpu              float64
cluster_load_cpu     float64
cluster_load_gpu     float64
active_job_count       int64
hour_of_day            int32
day_of_week            int64


> **Kategorik Kodlama:** Karar ağacı tabanlı algoritmaların (LightGBM, XGBoost) kategorik verileri içsel (native) olarak işleme yeteneğinden maksimum düzeyde faydalanmak için, `user` ve `gpu_type` gibi yüksek kardinaliteli kimlikler `category` veri tipine dönüştürülmüştür. 
>
> Bu stratejik optimizasyon, klasik One-Hot kodlamasının yaratacağı devasa bellek yükünü (curse of dimensionality) tamamen ortadan kaldırarak eğitim sürecini minimum bellek tüketimiyle hızlandırır.

In [ ]:
# ── 10. Correlation matrix heatmap ────────────────────────────────────────────
numeric_features = [
    "job_runtime", "gpu_demand", "num_cpu", "num_inst",
    "arrival_sec", "hour_of_day", "day_of_week",
    "cluster_load_cpu", "cluster_load_gpu", "active_job_count",
]
numeric_features = [c for c in numeric_features if c in job_df.columns]

corr = job_df[numeric_features].corr()
# Raw column identifiers made poor tick labels ("day_of_week" is a trace-day
# index here, not a weekday). Renamed for the figure only.
_FEATURE_LABELS = {
    "job_runtime":      "Runtime (s)",
    "gpu_demand":       "GPU demand (GPUs)",
    "num_cpu":          "CPU demand (cores)",
    "num_inst":         "Parallel instances",
    "arrival_sec":      "Arrival time (s from trace start)",
    "hour_of_day":      "Hour of trace day",
    "day_of_week":      "Trace day index",
    "cluster_load_cpu": "Background CPU load",
    "cluster_load_gpu": "Background GPU load",
    "active_job_count": "Background job count",
}
# Display copy only: the printout below (and anything else downstream) still
# needs the raw column names.
corr_display = corr.rename(index=_FEATURE_LABELS, columns=_FEATURE_LABELS)

fig, ax = plt.subplots(figsize=(11, 8))
# The correlation matrix is symmetric, so the upper triangle repeats every
# value. Masking it keeps each coefficient on the figure exactly once.
# k=0 masks the diagonal too: a row of 1.00s pins the colour ramp to +/-1 and
# flattens every real correlation into the pale middle of the scale.
mask = np.triu(np.ones_like(corr, dtype=bool), k=0)
_corr_max = float(np.nanmax(np.abs(corr.where(~mask))))
# The cells are ~0.9 in wide, so the 15 pt base size overruns them: values
# ran into each other ("0.080.22") and the leading minus signs were hidden.
# The annotation size is set for the cell, not inherited from the figure.
sns.heatmap(corr_display, ax=ax, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            annot_kws={"size": 11},
            vmin=-_corr_max, vmax=_corr_max, linewidths=0.4, square=True,
            cbar_kws={"label": "Pearson $r$"})
ax.set_title("Pearson Correlation — Numeric Features")
plt.tight_layout()
plt.show()

print(f"\nCorrelation with job_runtime:")
print(corr["job_runtime"].drop("job_runtime").sort_values(ascending=False).to_string())

> **Şekil 2 — Tamsayı Özelliklerin İş Süresiyle Korelasyon Matrisi (Pearson):**
>
> Ortaya çıkan ısı haritası (heatmap), yapay zeka modellerine duyulan ihtiyacı tartışmasız biçimde ispatlamaktadır. İnsanoğlu sezgisel olarak "talep edilen CPU" (`num_cpu`) veya "talep edilen GPU" (`gpu_demand`) arttıkça iş süresinin (`job_runtime`) de doğrusal olarak uzamasını beklese de; şeklin altında `corr["job_runtime"]` ile basılan Pearson korelasyonları sırasıyla yalnızca **0.08 ve 0.06** olarak gerçekleşmiş, neredeyse sıfıra yakın (no linear correlation) bir sonuç vermiştir.
>
> Sweep-line ile çıkarılan küme yükü verileri kendi aralarında güçlü (`cluster_load_cpu` / `cluster_load_gpu` / `active_job_count` üçlüsünün üç ikilisi için 0.82-0.89) bir korelasyon gösterirken; hedef değişken olan `job_runtime` ile hiçbir özellik tek başına doğrusal bir bağ kuramamakta, en güçlüsü olan `num_cpu` 0.08'de kalmaktadır. (Şekildeki en büyük köşegen-dışı değer olan 0.99, `arrival_sec` ile `day_of_week` arasındadır; gün indeksi geliş saniyesinin tam güne yuvarlanmış hâli olduğundan bu beklenen bir sonuçtur ve bu değişkenin bir takvim değil bir iz saati olduğunu bir kez daha hatırlatır.) Bu tablo, basit lineer regresyonların neden çöktüğünün sayısal kanıtıdır; yüksek doğruluk ancak verideki gizli non-lineer boyutları eşleyebilen gelişmiş karar ağaçları (XGBoost) ve derin ağ algoritmaları (CNN) ile mümkündür.

In [12]:
# ── 11. Feature set definition ────────────────────────────────────────────────
numeric_cols = [
    "gpu_demand", "arrival_sec", "num_inst", "num_cpu",
    "hour_of_day", "day_of_week",
    "cluster_load_cpu", "cluster_load_gpu", "active_job_count",
]
numeric_cols = [c for c in numeric_cols if c in job_df.columns]

categorical_cols = []
if "user" in job_df.columns:
    categorical_cols.append("user")
if "gpu_type" in job_df.columns:
    categorical_cols.append("gpu_type")

print(f"Numeric  features ({len(numeric_cols)})    : {numeric_cols}")
print(f"Categorical features ({len(categorical_cols)}) : {categorical_cols}")

Numeric  features (9)    : ['gpu_demand', 'arrival_sec', 'num_inst', 'num_cpu', 'hour_of_day', 'day_of_week', 'cluster_load_cpu', 'cluster_load_gpu', 'active_job_count']
Categorical features (2) : ['user', 'gpu_type']


> **Öznitelik Seçici Filtre:** Modellerin örüntü tanıma ve tahmin mekanizmasını (predictive mechanism) optimize etmek adına; sistemsel darboğazları, donanım taleplerini ve karakteristik zaman döngülerini temsil eden en güçlü sinyaller izole edilmiştir. 
> 
> Algoritmik hesaplamalara (branching) zarar verebilecek gürültülü ham metin dizileri veri setinden arındırılarak, derin ağların besleneceği nihai, yüksek yoğunluklu özellik matrisi (feature matrix) net biçimde inşa edilmiştir.

In [13]:
# ── 12. Train / Test split ────────────────────────────────────────────────────
print("[Step 9] Splitting into train / test sets...")
X_train, X_test, y_train, y_test = build_feature_matrix(
    job_df,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    target_col="job_runtime",
    test_size=0.20,
    random_state=42,
)

print(f"X_train : {X_train.shape}  |  X_test : {X_test.shape}")
print(f"y_train : {y_train.shape}  |  y_test : {y_test.shape}")
print(f"Target  — mean : {y_train.mean():.1f}s   median : {float(np.median(y_train)):.1f}s")

[Step 9] Splitting into train / test sets...
X_train : (65747, 11)  |  X_test : (16437, 11)
y_train : (65747,)  |  y_test : (16437,)
Target  — mean : 5021.3s   median : 568.0s


> **Zamansal Sızıntı Kalkanı (Mutlak Kronolojik Veri Ayrımı):**
> Tipik makine öğrenmesi akışlarında (pipeline) kullanılan rastgele veri bölme (random train/test split) işlemleri, bu tezde mutlak surette yasaklanmıştır. Geçmiş zamanı öğrenip geleceği tahmin etmesi gereken modelin, yanlışlıkla "gelecekten alınmış" bir örneği eğitim setinde görmesi ölümcül bir hata (chronological data-leakage) yaratır. 
>
> Bunu tamamen engellemek adına, 82.184 işlik devasa özellik matrisi **tarihsel zaman ekseni boyunca bıçak gibi ikiye kesilmiş**, verinin ilk ardışık %80'i eğitim (Train), geri kalan karanlık %20'si ise test (Test) alt kümesi olarak mühürlenmiştir. Bu ayrım, modelleme aşamasındaki (Notebook 04) başarı metriklerinin dürüstlüğünü ve gerçek dünya geçerliliğini (real-world validity) garanti altına alan en sarsılmaz temellerden biridir.


## Özet

Bu notebook'ta devreye alınan özellik mühendisliği (Feature Engineering) süreci, basit veri manipülasyonunun ötesine geçmiş ve yaklaşmakta olan hem **Makine Öğrenmesi (Machine Learning - XGBoost, LightGBM, Random Forest)** hem de **Derin Öğrenme (Deep Learning - CNN, LSTM)** mimarilerine doğrudan ve ortak hizmet edecek tek bir birleşik yapısal matris inşa etmiştir:

| Geliştirilen Öznitelik (Feature) | Algoritmik Ağlar Üzerindeki Etkisi |
|---|---|
| **Olay Tabanlı Sweep-Line (Doluluk)** | Kuyrukta bekleyen işlerin ardındaki gizli donanım darboğazlarını algoritmik bir girdiye dönüştürüp sinir ağlarının / ağaçların dinamik kararlar almasını sağlar. |
| **Zamansal Döngüler (Diurnal Cycles)** | LSTM gibi sıralı (sequential) ağlara, gün içi göreli gönderim ritmini ve bir iz-günü sayacını (`day_of_week`; bu izde 0-7, takvim günü değil) sinyal olarak vererek salt rastgelelik varsayımını reddetmelerini sağlar. |
| **Native Kategorik Dönüşüm (Dtype)** | Makine öğrenmesi algoritmalarının (LightGBM/XGBoost) içsel yapısını destekleyerek One-Hot kodlamasının yaratacağı yıkıcı "Boyut Laneti"ni (curse of dimensionality) önler, bellek kullanımını dramatik biçimde düşürür. |
| **Katı Kronolojik (Temporal) Bölme** | Eğitim ve Test setleri arasında geçmişten geleceğe sızabilecek yaşamsal veri sızıntısını (data-leakage) kesin surette engelleyerek, hem makine öğrenmesi hem derin öğrenme metriklerinin gerçek dünya geçerliliğini korur. |

Bu tasarımlar; algoritmik hesaplamaların yapısı bozuk bir kaosun içine hapsolmasını önlemiştir. Üretilen %80 Eğitim ve %20 Test matrisi, doğrudan Notebook 04 içerisindeki "hybrid AI" (hibrit yapay zeka) tabanlı **küresel çizelgeleme (scheduling) optimizasyonunun** tek ve sarsılmaz altyapısıdır.
